In [ ]:
# %pip install -r ../requirements.txt

In [ ]:
dbutils.library.restartPython()

In [ ]:
# Destino dos dados no volume do Unity Catalog.

import os
import sys

REPO_ROOT = os.path.dirname(os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

os.environ["PIPELINE_DATA_ROOT"] = "/Volumes/avaliacao_sicredi/novarota/dados"

print(f"raiz do projeto: {REPO_ROOT}")

In [ ]:
# Diagnostico do ambiente

from src.config import get_config_version, get_data_root, get_env, get_spark

spark = get_spark()

print(f"ambiente .......: {get_env()}")
print(f"raiz de dados ..: {get_data_root()}")
print(f"config version .: {get_config_version()}")
print(f"spark version ..: {spark.version}")
print(f"timezone .......: {spark.conf.get('spark.sql.session.timeZone')}")

In [ ]:
# Geração da massa
from scripts.gerar_massa import gerar_massa

gerar_massa()

In [ ]:
# Pipeline completo
from src.pipeline import executar

resumo = executar()

In [ ]:
# Conferência das tabelas materializadas
from src.config import resolve_target

for tabela in (
    "gold_transacoes_cartao",
    "gold_resumo_cliente_mes",
    "gold_ranking_estabelecimento",
    "gold_alertas_qualidade",
):
    linhas = spark.read.format("delta").load(resolve_target("gold", tabela)).count()
    print(f"{tabela:32s} {linhas:7d} linhas")

In [ ]:
# Consultas analíticas
from pathlib import Path

from src.transformations.gold import registrar_views, registrar_views_gold

registrar_views()
registrar_views_gold()

for arquivo in sorted(Path("sql/analises").glob("*.sql")):
    print(f"\n{'=' * 72}\n{arquivo.name}\n{'=' * 72}")
    display(spark.sql(arquivo.read_text(encoding="utf-8")))

In [ ]:
# Registro no catalogo
from src.config import resolve_target

CATALOGO = "avaliacao_sicredi"
SCHEMA = f"{CATALOGO}.novarota_gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")

for tabela in (
    "gold_transacoes_cartao",
    "gold_resumo_cliente_mes",
    "gold_ranking_estabelecimento",
    "gold_alertas_qualidade",
):
    (
        spark.read.format("delta")
        .load(resolve_target("gold", tabela))
        .write.mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SCHEMA}.{tabela}")
    )
    print(f"registrada: {SCHEMA}.{tabela}")

In [ ]:
display(
    spark.sql(f"""
        SELECT nome_estabelecimento, cidade_estabelecimento,
               valor_liquido, rank_valor
        FROM {CATALOGO}.novarota_gold.gold_ranking_estabelecimento
        ORDER BY rank_valor
        LIMIT 10
    """)
)